In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
import seaborn as sns


In [13]:
from pathlib import Path
import ssl
import urllib.request

cache_path = Path.home() / ".cache" / "seaborn-data" / "tips.csv"
cache_path.parent.mkdir(parents=True, exist_ok=True)

if not cache_path.exists():
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
    with urllib.request.urlopen(url, context=ssl._create_unverified_context()) as response:
        cache_path.write_bytes(response.read())

df = pd.read_csv(cache_path)
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  244 non-null    float64
 1   tip         244 non-null    float64
 2   sex         244 non-null    str    
 3   smoker      244 non-null    str    
 4   day         244 non-null    str    
 5   time        244 non-null    str    
 6   size        244 non-null    int64  
dtypes: float64(2), int64(1), str(4)
memory usage: 13.5 KB


In [15]:
df['sex'].value_counts()

sex
Male      157
Female     87
Name: count, dtype: int64

In [16]:
df['smoker'].value_counts()

smoker
No     151
Yes     93
Name: count, dtype: int64

In [19]:
X = df[['tip','sex','smoker','day','time','size']]
y = df['total_bill']

In [20]:
##Feature Encoding

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [21]:
X_train.head()

,tip,sex,smoker,day,time,size
115,3.50,Female,No,Sun,Dinner,2
181,5.65,Male,Yes,Sun,Dinner,2
225,2.50,Female,Yes,Fri,Lunch,2
68,2.01,Male,No,Sat,Dinner,2
104,4.08,Female,No,Sat,Dinner,2


In [22]:
#Feature Encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

In [23]:
le1 = LabelEncoder()
le2 = LabelEncoder()
le3 = LabelEncoder()


In [24]:
X_train['sex'] = le1.fit_transform(X_train['sex'])
X_train['smoker'] = le2.fit_transform(X_train['smoker'])
X_train['time'] = le3.fit_transform(X_train['time'])

In [25]:
X_train.head()

,tip,sex,smoker,day,time,size
115,3.50,0,0,Sun,0,2
181,5.65,1,1,Sun,0,2
225,2.50,0,1,Fri,1,2
68,2.01,1,0,Sat,0,2
104,4.08,0,0,Sat,0,2


In [26]:
X_test['sex'] = le1.transform(X_test['sex'])
X_test['smoker'] = le2.transform(X_test['smoker'])
X_test['time'] = le3.transform(X_test['time'])

In [30]:
#OneHotEncoding
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('onehot',OneHotEncoder(drop='first'),[3])],remainder='passthrough')

In [32]:
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

In [33]:
from sklearn.svm import SVR
model = SVR(kernel='linear')
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

In [34]:
from sklearn.metrics import r2_score, mean_absolute_error
print("R2 Score:",r2_score(y_test,y_pred))
print("MAE:",mean_absolute_error(y_test,y_pred))

R2 Score: 0.6036949191843275
MAE: 4.114837460698694


In [36]:
#Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV
parameters = {'kernel':['linear'],
              'C':[1,10,100],
              'gamma': [0.1,0.01,0.001]}
grid = GridSearchCV(estimator=model,param_grid=parameters)
grid.fit(X_train,y_train)
grid_pred = grid.predict(X_test)
print("Best Parameters:",grid.best_params_)

Best Parameters: {'C': 10, 'gamma': 0.1, 'kernel': 'linear'}


In [37]:
from sklearn.metrics import r2_score, mean_absolute_error
print("R2 Score:",r2_score(y_test,grid_pred))
print("MAE:",mean_absolute_error(y_test,grid_pred))

R2 Score: 0.5921477596048326
MAE: 4.173150683685804
